### PDF File Download via HTML page

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException, StaleElementReferenceException, ElementClickInterceptedException
import os
import time
import requests
from bs4 import BeautifulSoup

In [20]:
# output_dir = "pdf_extracted_Christina"
# output_dir = "pdf_extracted_Krista"
# output_dir = "pdf_extracted_Richard"
output_dir = "pdf_extracted_Silvia"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [4]:
base_url = "https://scholarworks.indianapolis.iu.edu"

In [12]:
## Approach
'''
Downloading the file
sending requests with wait time of 10 secs

txt file --> txt/plain (MIME) Content-type else pdf
file_name appending name and its type

saving the unique files in output dir to avoid overwritting

by using the name using splittext and assiging a counter to track the files
writing the pdfs files since it written in raw format
'''
def download_file(file_link, file_name):
    try:
        response = requests.get(file_link, timeout=10, stream=True)
        response.raise_for_status()

        content_type = response.headers.get("Content-Type", "").lower()
        if "text/plain" in content_type:
            file_extension = ".txt"
        elif "application/pdf" in content_type:
            file_extension = ".pdf"
        else:
            print(f"Unsupported file type ({content_type}) for {file_link}. Skipping...")
            return None

        base_file_name = os.path.splitext(file_name)[0]
        file_name = base_file_name + file_extension

        unique_file_name = os.path.join(output_dir, file_name)
        counter = 1
        while os.path.exists(unique_file_name):
            unique_file_name = os.path.join(output_dir, f"{base_file_name}_{counter}{file_extension}")
            counter += 1
    
        with open(unique_file_name, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024):
                if chunk:  # filter out keep-alive new chunks
                    f.write(chunk)
        print(f"Downloaded: {unique_file_name}")
        return unique_file_name

    except requests.exceptions.RequestException as e:
        print(f"Error downloading file: {file_link}, Error: {e}")
        return None

In [9]:
### Approach
"""
Via HTML parser extracting the file read in the path

logic 
looking for all anchor tags
setting href to anchor with href

tag identification for tags (downloads and bitstream)
1. url logic conditions
2.  Text file indication in span tag
3. Pdf file indication in span tag
"""


def extract_pdf_links(html_path):
    with open(html_path, "r", encoding="utf-8") as file:
        html_content = file.read()
    soup = BeautifulSoup(html_content, "html.parser")

    pdf_links = set()
    for anchor in soup.find_all("a", href=True):
        href = anchor['href']
        # Check for 'download' in the href to ensure it's a direct content link
        if "bitstreams" in href and "download" in href:
            full_url = base_url + href if href.startswith("/") else href
            file_name = href.split("/")[-2] + ".pdf"
            pdf_links.add((full_url, file_name))

        elif anchor.find("span", text=lambda x: x and "txt" in x.lower()):
            # Check for nested <span> containing 'txt'
            full_url = base_url + href if href.startswith("/") else href
            file_name = href.split("/")[-2] + ".txt" if "bitstreams" in href else href.split("/")[-1]
            pdf_links.add((full_url, file_name))

        else:
            span = anchor.find("span", text=lambda x: x and "pdf" in x.lower())
            if span:
                full_url = base_url + href if href.startswith("/") else href
                file_name = span.text.strip().replace(" ", "_") + ".pdf"                ####
                pdf_links.add((full_url, file_name))

    print(f"Found {len(pdf_links)} unique PDF links in {html_path}")
    return list(pdf_links)

In [14]:
# approach
'''
if dir:
check for directory

else:
file ends with html
joining the pdf file path
calling the (extract_pdf_links)
calling the (download_pdf)
'''

def process_html_directory(html_dir):
    """Process all HTML files in a directory to download PDFs."""
    # edge case
    if not os.path.exists(html_dir):
        print(f"Error: Directory {html_dir} does not exist.")
        return

    for html_file in os.listdir(html_dir):
        if html_file.endswith(".html"):
            html_path = os.path.join(html_dir, html_file)
            print(f"Processing HTML file: {html_file}")

            # Extract PDF links and download them
            pdf_links = extract_pdf_links(html_path)
            for pdf_link, file_name in pdf_links:

                unique_file_name = f"{os.path.splitext(html_file)[0]}_{file_name}"
                download_file(pdf_link, unique_file_name)

In [21]:
def main():
    # html_dir = "html_extracted_Christina"
    # html_dir = "html_extracted_Krista"
    # html_dir = "html_extracted_Richard"
    html_dir = "html_extracted_Silvia"
    process_html_directory(html_dir)

if __name__ == "__main__":
    main()

Processing HTML file: 0a83dc8e-cd6f-4d9d-af9b-f195187ad817.html
Found 1 unique PDF links in html_extracted_Silvia\0a83dc8e-cd6f-4d9d-af9b-f195187ad817.html


C:\Users\kirth\AppData\Local\Temp\ipykernel_8872\734058891.py:30: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  elif anchor.find("span", text=lambda x: x and "txt" in x.lower()):
C:\Users\kirth\AppData\Local\Temp\ipykernel_8872\734058891.py:37: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  span = anchor.find("span", text=lambda x: x and "pdf" in x.lower())


Downloaded: pdf_extracted_Silvia\0a83dc8e-cd6f-4d9d-af9b-f195187ad817_e2c94177-e1c5-4ceb-a9f6-bd9928f03fc9.pdf
Processing HTML file: 0dc66297-26c0-44d4-909e-66832109cf2a.html
Found 1 unique PDF links in html_extracted_Silvia\0dc66297-26c0-44d4-909e-66832109cf2a.html
Downloaded: pdf_extracted_Silvia\0dc66297-26c0-44d4-909e-66832109cf2a_2bb62a7e-a3cd-4480-9803-599e440b4a79.pdf
Processing HTML file: 0eb5b96e-663e-410d-a683-eeb73e1756c8.html
Found 1 unique PDF links in html_extracted_Silvia\0eb5b96e-663e-410d-a683-eeb73e1756c8.html
Downloaded: pdf_extracted_Silvia\0eb5b96e-663e-410d-a683-eeb73e1756c8_15aa67d8-1c83-4190-8419-71cd3636e859.pdf
Processing HTML file: 0f28f46a-9060-4314-bd73-6c0c649d9c69.html
Found 1 unique PDF links in html_extracted_Silvia\0f28f46a-9060-4314-bd73-6c0c649d9c69.html
Downloaded: pdf_extracted_Silvia\0f28f46a-9060-4314-bd73-6c0c649d9c69_e35c6548-9f4a-452c-a337-18376ec85ae3.pdf
Processing HTML file: 1136b4f0-7789-4838-b9b3-3a3c676b783a.html
Found 1 unique PDF links